# Microkinetic model for the oxygen reduction reaction (ORR) on FePc.

This notebook builds a complete microkinetic model for the ORR on FePc without running any new atomistic calculations. Instead, it uses experimentally informed parameters and the same five-step surface cycle (O₂ adsorption, then four sequential electron-transfer steps through OOH*, O*, OH*, back to the free site) that underlies the Computational Hydrogen Electrode framework. The thermodynamic parameters (G_OH, b_OOH, b_O) define the free energy landscape. The kinetic parameters (five rate constants and an equilibrium constant for O₂ adsorption) define how fast each step runs. Electrochemical Butler-Volmer rate constants are derived from the thermodynamic levels and the applied potential, so the model is internally self-consistent.

In [ ]:
# Cell 1: Imports and plotting settings

import numpy as np
import matplotlib.pyplot as plt

from scipy.linalg import solve
from ipywidgets import interact, FloatSlider, FloatLogSlider, fixed
from IPython.display import display

l = 8.25/2.54 # inches
plt.rc('figure',figsize=(l,l)) # figure size
plt.rc('figure',      dpi=100) # figure dpi
plt.rc('lines',  linewidth=4)  # linewidth
plt.rc('font',        size=10) # fontsize of the text
plt.rc('font', family='sans-serif') # font family
plt.rc('axes',   titlesize=10) # fontsize of the axes title
plt.rc('axes',   labelsize=9)  # fontsize of the x and y labels
plt.rc('xtick',  labelsize=8)  # fontsize of the tick labels
plt.rc('ytick',  labelsize=8)  # fontsize of the tick labels
plt.rc('legend', fontsize =8)  # fontsize of the legend
plt.rc('figure', titlesize=10) # fontsize of the figure title
# Tol's color scheme
colors = ['#1965B0', '#7BAFDE', '#90C987', '#F1932D', '#DC050C']

# Cell 2: Parameters and free-energy diagram

We use a simplified ORR free-energy path:

$$
\mathrm{\ast + O_2}
\rightarrow
\mathrm{OOH\ast}
\rightarrow
\mathrm{O\ast}
\rightarrow
\mathrm{OH\ast}
\rightarrow
\mathrm{\ast + H_2O}
$$

The final state is set to zero:

$$
G(\mathrm{\ast + H_2O}) = 0
$$

The initial state at zero potential is

$$
G(\mathrm{\ast + O_2}) = 4.92\ \mathrm{eV}
$$

because

$$
4.92 = 4 \times 1.23
$$

The adjustable thermodynamic parameters are:

$$
G_{\mathrm{OH}}
$$

$$
b_{\mathrm{OOH}}
$$

$$
b_{\mathrm{O}}
$$

with

$$
G_{\mathrm{OOH}} = G_{\mathrm{OH}} + b_{\mathrm{OOH}}
$$

and

$$
G_{\mathrm{O}} = 2(G_{\mathrm{OH}} - b_{\mathrm{O}})
$$

At potential $U$, the free-energy levels are:

$$
G_0 = 4.92 - 4U
$$

$$
G_1 = G_{\mathrm{OOH}} - 3U
$$

$$
G_2 = G_{\mathrm{O}} - 2U
$$

$$
G_3 = G_{\mathrm{OH}} - U
$$

$$
G_4 = 0
$$

In [ ]:
# Cell 2: Parameters and free-energy diagram

# Constants
F = 96485.0
R = 8.314
T = 295.15
V_T = R * T / F

E_ORR = 1.23
G_ORR = 4.0 * E_ORR

default_params = {
    # Thermodynamics
    "G_OH": 0.82,       # eV
    "b_OOH": 3.03,      # eV
    "b_O": -0.30,       # eV

    # Kinetics for later cells
    "k_ads": 2000.0,    # s-1
    "K_ads": 0.6,
    "k_OOH": 3.0,       # s-1
    "k_O": 0.01,        # s-1
    "k_OH": 0.01,       # s-1
    "k_free": 60.0,     # s-1

    # Active-site concentration
    "Gamma": 1.2e-10,   # mol cm-2
}

def intermediate_energies(G_OH, b_OOH, b_O):
    """
    Return adsorption energies:
        G_OOH, G_O, G_OH
    """
    G_OOH = G_OH + b_OOH
    G_O = 2.0 * (G_OH - b_O)

    return [G_OOH, G_O, G_OH]

def free_energy_states(U, Es):
    """
    Return the five free-energy levels:
        * + O2, OOH*, O*, OH*, * + H2O
    """
    return [
        G_ORR - 4.0 * U,
        Es[0] - 3.0 * U,
        Es[1] - 2.0 * U,
        Es[2] - 1.0 * U,
        0.0,
    ]

def plot_at_U(U, Es, color="#0000cc", label="model"):
    """
    Plot one free-energy diagram at potential U.
    """
    states = free_energy_states(U, Es)

    bar_x = [
        (-0.2, 0.2),
        (0.8, 1.2),
        (1.8, 2.2),
        (2.8, 3.2),
        (3.8, 4.2),
    ]

    for i, (x0, x1) in enumerate(bar_x):
        plt.plot(
            [x0, x1],
            [states[i], states[i]],
            c=color,
            linestyle="-",
            linewidth=3,
            label=f"{label}, U = {U:.2f} V" if i == 0 else None,
        )

    for i in range(4):
        plt.plot(
            [bar_x[i][1], bar_x[i + 1][0]],
            [states[i], states[i + 1]],
            c=color,
            linestyle=":",
            linewidth=1,
        )

def plot_free_energy_diagram(G_OH, b_OOH, b_O):
    """
    Plot free-energy diagrams at U = 1.23 V, U = U_L, and U = 0 V.
    """
    Es = intermediate_energies(G_OH, b_OOH, b_O)

    step_energies_at_U0 = [
        G_ORR - Es[0],
        Es[0] - Es[1],
        Es[1] - Es[2],
        Es[2],
    ]

    U_L = E_ORR - min(step_energies_at_U0)

    plt.figure()

    plot_at_U(1.23, Es, colors[1], "Cat.")
    plot_at_U(round(1.23 - U_L, 2), Es, colors[2], "Cat.")
    plot_at_U(0.00, Es, colors[0], "Cat.")

    plt.xlabel("Reaction coordinate")
    plt.ylabel("Free energies / eV")
    plt.xticks(
        [0, 1, 2, 3, 4],
        ["*+O$_2$", "OOH*", "O*", "OH*", "*+H$_2$O"],
    )
    plt.ylim(-1, 5.5)
    plt.legend()
    plt.show()

interact(
    plot_free_energy_diagram,
    G_OH=FloatSlider(
        value=default_params["G_OH"],
        min=0.50,
        max=1.10,
        step=0.01,
        description="G_OH / eV",
        continuous_update=False,
    ),
    b_OOH=FloatSlider(
        value=default_params["b_OOH"],
        min=2.50,
        max=3.50,
        step=0.01,
        description="b_OOH / eV",
        continuous_update=False,
    ),
    b_O=FloatSlider(
        value=default_params["b_O"],
        min=-0.80,
        max=0.20,
        step=0.01,
        description="b_O / eV",
        continuous_update=False,
    ),
);

interactive(children=(FloatSlider(value=0.82, continuous_update=False, description='G_OH / eV', max=1.1, min=0…

# Cell 3: Steady-state coverages

The kinetic model uses the full surface cycle:

$$
1:\quad \mathrm{\ast + O_2}
\rightleftharpoons
\mathrm{O_2\ast}
$$

$$
2:\quad \mathrm{O_2\ast}
\rightleftharpoons
\mathrm{OOH\ast}
$$

$$
3:\quad \mathrm{OOH\ast}
\rightleftharpoons
\mathrm{O\ast}
$$

$$
4:\quad \mathrm{O\ast}
\rightleftharpoons
\mathrm{OH\ast}
$$

$$
5:\quad \mathrm{OH\ast}
\rightleftharpoons
\mathrm{\ast}
$$

The surface coverages are

$$
\theta_{\ast},
\theta_{\mathrm{O_2}},
\theta_{\mathrm{OOH}},
\theta_{\mathrm{O}},
\theta_{\mathrm{OH}}
$$

with the site balance

$$
\theta_{\ast}
+
\theta_{\mathrm{O_2}}
+
\theta_{\mathrm{OOH}}
+
\theta_{\mathrm{O}}
+
\theta_{\mathrm{OH}}
=
1
$$

The thermodynamic parameters are the same as in the free-energy diagram:

$$
G_{\mathrm{OOH}} = G_{\mathrm{OH}} + b_{\mathrm{OOH}}
$$

$$
G_{\mathrm{O}} = 2(G_{\mathrm{OH}} - b_{\mathrm{O}})
$$

The hidden reversible potentials used in the rate constants are calculated from the free-energy differences between neighboring states.

In [ ]:
# Cell 3: Steady-state coverages

coverage_labels = [
    r"$\theta_{\ast}$",
    r"$\theta_{\mathrm{O_2}}$",
    r"$\theta_{\mathrm{OOH}}$",
    r"$\theta_{\mathrm{O}}$",
    r"$\theta_{\mathrm{OH}}$",
]

def thermodynamic_levels(G_OH, b_OOH, b_O, K_ads):
    """
    Energies at U = 0 for the kinetic cycle:
        *+O2, O2*, OOH*, O*, OH*, *
    """
    G_OOH = G_OH + b_OOH
    G_O = 2.0 * (G_OH - b_O)
    dG_ads = -V_T * np.log(K_ads)
    G_O2 = G_ORR + dG_ads

    return {
        "G_O2": G_O2,
        "G_OOH": G_OOH,
        "G_O": G_O,
        "G_OH": G_OH,
    }

def reversible_potentials(G_OH, b_OOH, b_O, K_ads):
    """
    Reversible potentials for electron-transfer steps 2-5.
    """
    G = thermodynamic_levels(G_OH, b_OOH, b_O, K_ads)

    return {
        "E_OOH": G["G_O2"] - G["G_OOH"],
        "E_O": G["G_OOH"] - G["G_O"],
        "E_OH": G["G_O"] - G["G_OH"],
        "E_free": G["G_OH"],
    }

def electrochemical_rate(k0, U, U_eq, alpha=0.5):
    """
    Forward and reverse rate constants for one electron-transfer step.
    """
    k_forward = k0 * np.exp(-alpha * (U - U_eq) / V_T)
    k_reverse = k0 * np.exp((1.0 - alpha) * (U - U_eq) / V_T)

    return k_forward, k_reverse

def rate_constants(
    U,
    G_OH,
    b_OOH,
    b_O,
    k_ads,
    K_ads,
    k_OOH,
    k_O,
    k_OH,
    k_free,
):
    """
    Rate constants for the natural ORR flow:
        1 adsorption
        2 OOH formation
        3 O formation
        4 OH formation
        5 free-site regeneration
    """
    U_eq = reversible_potentials(G_OH, b_OOH, b_O, K_ads)

    k1f = k_ads
    k1r = k_ads / K_ads

    k2f, k2r = electrochemical_rate(k_OOH, U, U_eq["E_OOH"])
    k3f, k3r = electrochemical_rate(k_O, U, U_eq["E_O"])
    k4f, k4r = electrochemical_rate(k_OH, U, U_eq["E_OH"])
    k5f, k5r = electrochemical_rate(k_free, U, U_eq["E_free"])

    return k1f, k1r, k2f, k2r, k3f, k3r, k4f, k4r, k5f, k5r

def steady_state_coverages(
    U,
    G_OH,
    b_OOH,
    b_O,
    k_ads,
    K_ads,
    k_OOH,
    k_O,
    k_OH,
    k_free,
):
    """
    Solve coverages at one potential.

    Coverage order:
        *, O2*, OOH*, O*, OH*
    """
    k1f, k1r, k2f, k2r, k3f, k3r, k4f, k4r, k5f, k5r = rate_constants(
        U, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free
    )

    M = np.array([
        [k1f, -(k1r + k2f), k2r, 0.0, 0.0],
        [0.0, k2f, -(k2r + k3f), k3r, 0.0],
        [0.0, 0.0, k3f, -(k3r + k4f), k4r],
        [k5r, 0.0, 0.0, k4f, -(k4r + k5f)],
        [1.0, 1.0, 1.0, 1.0, 1.0],
    ])

    rhs = np.array([0.0, 0.0, 0.0, 0.0, 1.0])

    return solve(M, rhs)

def coverage_curves(
    U_grid,
    G_OH,
    b_OOH,
    b_O,
    k_ads,
    K_ads,
    k_OOH,
    k_O,
    k_OH,
    k_free,
):
    theta = np.zeros((len(U_grid), 5))

    for i, U in enumerate(U_grid):
        theta[i] = steady_state_coverages(
            U, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free
        )

    return theta

def plot_coverages(
    G_OH,
    b_OOH,
    b_O,
    k_ads,
    K_ads,
    k_OOH,
    k_O,
    k_OH,
    k_free,
):
    U_grid = np.linspace(1.0, 0.2, 500)

    theta = coverage_curves(
        U_grid, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free
    )

    plt.figure()

    for i, label in enumerate(coverage_labels):
        plt.plot(U_grid, theta[:, i], label=label)

    plt.xlabel("U / V vs RHE")
    plt.ylabel("Coverage")
    plt.ylim(0, 1)
    plt.legend()
    plt.show()

interact(
    plot_coverages,
    G_OH=FloatSlider(
        value=default_params["G_OH"],
        min=0.50,
        max=1.10,
        step=0.01,
        description="G_OH",
        continuous_update=False,
    ),
    b_OOH=FloatSlider(
        value=default_params["b_OOH"],
        min=2.50,
        max=3.50,
        step=0.01,
        description="b_OOH",
        continuous_update=False,
    ),
    b_O=FloatSlider(
        value=default_params["b_O"],
        min=-0.80,
        max=0.20,
        step=0.01,
        description="b_O",
        continuous_update=False,
    ),
    k_ads=FloatLogSlider(
        value=default_params["k_ads"],
        base=10,
        min=1,
        max=5,
        step=0.1,
        description="k_ads",
        continuous_update=False,
    ),
    K_ads=FloatSlider(
        value=default_params["K_ads"],
        min=0.05,
        max=3.0,
        step=0.05,
        description="K_ads",
        continuous_update=False,
    ),
    k_OOH=FloatLogSlider(
        value=default_params["k_OOH"],
        base=10,
        min=-2,
        max=3,
        step=0.1,
        description="k_OOH",
        continuous_update=False,
    ),
    k_O=FloatLogSlider(
        value=default_params["k_O"],
        base=10,
        min=-3,
        max=3,
        step=0.1,
        description="k_O",
        continuous_update=False,
    ),
    k_OH=FloatLogSlider(
        value=default_params["k_OH"],
        base=10,
        min=-3,
        max=3,
        step=0.1,
        description="k_OH",
        continuous_update=False,
    ),
    k_free=FloatLogSlider(
        value=default_params["k_free"],
        base=10,
        min=-2,
        max=4,
        step=0.1,
        description="k_free",
        continuous_update=False,
    ),
);

interactive(children=(FloatSlider(value=0.82, continuous_update=False, description='G_OH', max=1.1, min=0.5, s…

# Cell 4: Kinetic current

The kinetic current is calculated from the electron-transfer steps in the surface cycle:

$$
1:\quad \mathrm{\ast + O_2}
\rightleftharpoons
\mathrm{O_2\ast}
$$

$$
2:\quad \mathrm{O_2\ast}
\rightleftharpoons
\mathrm{OOH\ast}
$$

$$
3:\quad \mathrm{OOH\ast}
\rightleftharpoons
\mathrm{O\ast}
$$

$$
4:\quad \mathrm{O\ast}
\rightleftharpoons
\mathrm{OH\ast}
$$

$$
5:\quad \mathrm{OH\ast}
\rightleftharpoons
\mathrm{\ast}
$$

Step 1 is chemical oxygen adsorption. Steps 2–5 transfer one electron each.

The kinetic current density is

$$
j_k =
-F\Gamma(r_2+r_3+r_4+r_5)
$$

Here $\Gamma$ is the active-site concentration. In the slider we use units of

$$
\mathrm{pmol\ cm^{-2}}
$$

so the code converts it as

$$
\Gamma_{\mathrm{mol}} =
\Gamma_{\mathrm{pmol}}\times 10^{-12}
$$

In [ ]:
# Cell 4: Kinetic current

def reaction_rates(U, theta, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free):
    """
    Return reaction rates r1-r5 for the natural ORR flow.

    Coverage order:
        *, O2*, OOH*, O*, OH*
    """
    theta_free, theta_O2, theta_OOH, theta_O, theta_OH = theta

    k1f, k1r, k2f, k2r, k3f, k3r, k4f, k4r, k5f, k5r = rate_constants(
        U, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free
    )

    r1 = k1f * theta_free - k1r * theta_O2
    r2 = k2f * theta_O2 - k2r * theta_OOH
    r3 = k3f * theta_OOH - k3r * theta_O
    r4 = k4f * theta_O - k4r * theta_OH
    r5 = k5f * theta_OH - k5r * theta_free

    return np.array([r1, r2, r3, r4, r5])


def kinetic_current_density(
    U,
    G_OH,
    b_OOH,
    b_O,
    k_ads,
    K_ads,
    k_OOH,
    k_O,
    k_OH,
    k_free,
    gamma,
):
    """
    Return kinetic current density in A cm-2.

    gamma:
        Active-site concentration in mol cm-2.
    """
    theta = steady_state_coverages(
        U, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free
    )

    r = reaction_rates(
        U, theta, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free
    )

    return -F * gamma * np.sum(r[1:])


def kinetic_current_curve(
    U_grid,
    G_OH,
    b_OOH,
    b_O,
    k_ads,
    K_ads,
    k_OOH,
    k_O,
    k_OH,
    k_free,
    gamma,
):
    return np.array([
        kinetic_current_density(
            U, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free, gamma
        )
        for U in U_grid
    ])


def plot_kinetic_current(
    G_OH,
    b_OOH,
    b_O,
    k_ads,
    K_ads,
    k_OOH,
    k_O,
    k_OH,
    k_free,
    gamma,
):
    gamma_mol = gamma * 1e-12
    U_grid = np.linspace(1.0, 0.2, 500)

    jk = kinetic_current_curve(
        U_grid, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free, gamma_mol
    )

    plt.figure()
    plt.plot(U_grid, 1000.0 * jk, color=colors[0])

    plt.xlabel("U / V vs RHE")
    plt.ylabel(r"$j_k$ / mA cm$^{-2}$")
    plt.ylim(-10, 0)
    plt.show()


interact(
    plot_kinetic_current,
    G_OH=FloatSlider(
        value=default_params["G_OH"],
        min=0.50,
        max=1.10,
        step=0.01,
        description="G_OH",
        continuous_update=False,
    ),
    b_OOH=FloatSlider(
        value=default_params["b_OOH"],
        min=2.50,
        max=3.50,
        step=0.01,
        description="b_OOH",
        continuous_update=False,
    ),
    b_O=FloatSlider(
        value=default_params["b_O"],
        min=-0.80,
        max=0.20,
        step=0.01,
        description="b_O",
        continuous_update=False,
    ),
    k_ads=FloatLogSlider(
        value=default_params["k_ads"],
        base=10,
        min=1,
        max=5,
        step=0.1,
        description="k_ads",
        continuous_update=False,
    ),
    K_ads=FloatSlider(
        value=default_params["K_ads"],
        min=0.05,
        max=3.0,
        step=0.05,
        description="K_ads",
        continuous_update=False,
    ),
    k_OOH=FloatLogSlider(
        value=default_params["k_OOH"],
        base=10,
        min=-2,
        max=3,
        step=0.1,
        description="k_OOH",
        continuous_update=False,
    ),
    k_O=FloatLogSlider(
        value=default_params["k_O"],
        base=10,
        min=-3,
        max=3,
        step=0.1,
        description="k_O",
        continuous_update=False,
    ),
    k_OH=FloatLogSlider(
        value=default_params["k_OH"],
        base=10,
        min=-3,
        max=3,
        step=0.1,
        description="k_OH",
        continuous_update=False,
    ),
    k_free=FloatLogSlider(
        value=default_params["k_free"],
        base=10,
        min=-2,
        max=4,
        step=0.1,
        description="k_free",
        continuous_update=False,
    ),
    gamma=FloatSlider(
        value=default_params["Gamma"] / 1e-12,
        min=10.0,
        max=2000.0,
        step=10.0,
        description="gamma",
        continuous_update=False,
    ),
);

interactive(children=(FloatSlider(value=0.82, continuous_update=False, description='G_OH', max=1.1, min=0.5, s…

# Cell 5: RRDE disk current

The kinetic model gives the current density controlled only by surface reaction rates:

$$
j_k
$$

In an RRDE experiment, the measured disk current is also limited by oxygen transport to the rotating disk.

The diffusion-limited current density is estimated with the Levich equation:

$$
j_L =
-0.62 n F D_{\mathrm{O_2}}^{2/3}
\nu^{-1/6}
C_{\mathrm{O_2}}
\omega^{1/2}
$$

where $n=4$ for four-electron ORR, $D_{\mathrm{O_2}}$ is the oxygen diffusion coefficient, $\nu$ is the kinematic viscosity, $C_{\mathrm{O_2}}$ is the bulk oxygen concentration, and $\omega$ is the rotation rate in rad s$^{-1}$:

$$
\omega = \frac{2\pi\ \mathrm{rpm}}{60}
$$

The kinetic and transport limits are combined with the Koutecky-Levich relation:

$$
\frac{1}{j_{\mathrm{disk}}}
=
\frac{1}{j_k}
+
\frac{1}{j_L}
$$

The plotted current is cathodic, so it is negative.

In [ ]:
# Cell 5: RRDE disk current

from ipywidgets import Dropdown

def levich_current_density(rpm, n=4, D_O2=2.0e-5, nu=0.01, C_O2=1.2e-6):
    """
    Levich diffusion-limited current density in A cm-2.
    """
    omega = 2.0 * np.pi * rpm / 60.0

    return (
        -0.62
        * n
        * F
        * D_O2 ** (2.0 / 3.0)
        * nu ** (-1.0 / 6.0)
        * C_O2
        * omega ** 0.5
    )


def disk_current_density(jk, jL):
    """
    Koutecky-Levich combination.

    Uses magnitudes internally and returns a cathodic current.
    """
    jk_abs = np.maximum(np.abs(jk), 1e-30)
    jL_abs = max(abs(jL), 1e-30)

    j_disk_abs = 1.0 / (1.0 / jk_abs + 1.0 / jL_abs)

    return -j_disk_abs


def plot_rrde_disk_current(
    G_OH,
    b_OOH,
    b_O,
    k_ads,
    K_ads,
    k_OOH,
    k_O,
    k_OH,
    k_free,
    gamma,
    rpm,
):
    gamma_mol = gamma * 1e-12
    U_grid = np.linspace(1.0, 0.2, 500)

    jk = kinetic_current_curve(
        U_grid, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free, gamma_mol
    )

    jL = levich_current_density(rpm)
    j_disk = disk_current_density(jk, jL)

    plt.figure()
    plt.plot(U_grid, 1000.0 * j_disk, color=colors[0])
    plt.axhline(1000.0 * jL, color=colors[4], linestyle=":", linewidth=1)

    plt.xlabel("U / V vs RHE")
    plt.ylabel(r"$j_{\mathrm{disk}}$ / mA cm$^{-2}$")
    plt.ylim(-10, 0)
    plt.show()

    print(f"rpm = {rpm}")
    print(f"sqrt(rpm) = {np.sqrt(rpm):.0f}")
    print(f"j_L = {1000.0 * jL:.3f} mA cm-2")
    print(f"most cathodic j_k = {1000.0 * np.min(jk):.3f} mA cm-2")


interact(
    plot_rrde_disk_current,
    G_OH=FloatSlider(
        value=default_params["G_OH"],
        min=0.50,
        max=1.10,
        step=0.01,
        description="G_OH",
        continuous_update=False,
    ),
    b_OOH=FloatSlider(
        value=default_params["b_OOH"],
        min=2.50,
        max=3.50,
        step=0.01,
        description="b_OOH",
        continuous_update=False,
    ),
    b_O=FloatSlider(
        value=default_params["b_O"],
        min=-0.80,
        max=0.20,
        step=0.01,
        description="b_O",
        continuous_update=False,
    ),
    k_ads=FloatLogSlider(
        value=default_params["k_ads"],
        base=10,
        min=1,
        max=5,
        step=0.1,
        description="k_ads",
        continuous_update=False,
    ),
    K_ads=FloatSlider(
        value=default_params["K_ads"],
        min=0.05,
        max=3.0,
        step=0.05,
        description="K_ads",
        continuous_update=False,
    ),
    k_OOH=FloatLogSlider(
        value=default_params["k_OOH"],
        base=10,
        min=-2,
        max=3,
        step=0.1,
        description="k_OOH",
        continuous_update=False,
    ),
    k_O=FloatLogSlider(
        value=default_params["k_O"],
        base=10,
        min=-3,
        max=3,
        step=0.1,
        description="k_O",
        continuous_update=False,
    ),
    k_OH=FloatLogSlider(
        value=default_params["k_OH"],
        base=10,
        min=-3,
        max=3,
        step=0.1,
        description="k_OH",
        continuous_update=False,
    ),
    k_free=FloatLogSlider(
        value=default_params["k_free"],
        base=10,
        min=-2,
        max=4,
        step=0.1,
        description="k_free",
        continuous_update=False,
    ),
    gamma=FloatSlider(
        value=default_params["Gamma"] / 1e-12,
        min=10.0,
        max=2000.0,
        step=10.0,
        description="gamma",
        continuous_update=False,
    ),
    rpm=Dropdown(
        options=[400, 900, 1600, 2500, 3600],
        value=1600,
        description="rpm",
    ),
);

interactive(children=(FloatSlider(value=0.82, continuous_update=False, description='G_OH', max=1.1, min=0.5, s…

# Cell 6: Summary dashboard

We now combine the three main outputs in one interactive dashboard.

The same parameters update:

1. the free-energy diagram,
2. the RDE disk current,
3. the steady-state coverages.

The thermodynamic sliders control the relative energies:

$$
G_{\mathrm{OH}},\quad b_{\mathrm{OOH}},\quad b_{\mathrm{O}}
$$

The kinetic sliders control the surface reaction rates:

$$
k_{\mathrm{ads}},\quad k_{\mathrm{OOH}},\quad k_{\mathrm{O}},\quad k_{\mathrm{OH}},\quad k_{\mathrm{free}}
$$

The RDE parameters are the active-site concentration

$$
\Gamma
$$

in pmol cm$^{-2}$, and the rotation rate in rpm.

The disk current is calculated from

$$
\frac{1}{j_{\mathrm{disk}}}
=
\frac{1}{j_k}
+
\frac{1}{j_L}
$$

In [ ]:
# Cell 6: Summary dashboard

def draw_free_energy_on_axis(ax, G_OH, b_OOH, b_O):
    Es = intermediate_energies(G_OH, b_OOH, b_O)

    step_energies_at_U0 = [
        G_ORR - Es[0],
        Es[0] - Es[1],
        Es[1] - Es[2],
        Es[2],
    ]

    U_L = E_ORR - min(step_energies_at_U0)
    U_values = [1.23, round(1.23 - U_L, 2), 0.00]
    U_colors = [colors[1], colors[2], colors[0]]

    bar_x = [
        (-0.2, 0.2),
        (0.8, 1.2),
        (1.8, 2.2),
        (2.8, 3.2),
        (3.8, 4.2),
    ]

    for U, color in zip(U_values, U_colors):
        states = free_energy_states(U, Es)

        for i, (x0, x1) in enumerate(bar_x):
            ax.plot(
                [x0, x1],
                [states[i], states[i]],
                c=color,
                linestyle="-",
                linewidth=3,
                label=f"U = {U:.2f} V" if i == 0 else None,
            )

        for i in range(4):
            ax.plot(
                [bar_x[i][1], bar_x[i + 1][0]],
                [states[i], states[i + 1]],
                c=color,
                linestyle=":",
                linewidth=1,
            )

    ax.set_xlabel("Reaction coordinate")
    ax.set_ylabel("Free energies / eV")
    ax.set_xticks([0, 1, 2, 3, 4])
    ax.set_xticklabels(["*+O$_2$", "OOH*", "O*", "OH*", "*+H$_2$O"])
    ax.set_ylim(-1, 5.5)
    ax.legend()
    ax.set_title("Free-energy diagram")


def half_wave_potential(x, y):
    y_start = y[0]
    y_end = y[-1]
    y_half = 0.5 * (y_start + y_end)

    x_half = None

    for i in range(len(x) - 1):
        y0 = y[i]
        y1 = y[i + 1]

        if (y0 - y_half) * (y1 - y_half) <= 0:
            if y1 == y0:
                x_half = 0.5 * (x[i] + x[i + 1])
            else:
                x_half = x[i] + (y_half - y0) * (x[i + 1] - x[i]) / (y1 - y0)
            break

    return x_half, y_half


def plot_summary_dashboard(
    G_OH,
    b_OOH,
    b_O,
    k_ads,
    K_ads,
    k_OOH,
    k_O,
    k_OH,
    k_free,
    gamma,
    rpm,
):
    gamma_mol = gamma * 1e-12
    U_grid = np.linspace(1.0, 0.2, 500)

    theta = coverage_curves(
        U_grid, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free
    )

    jk = kinetic_current_curve(
        U_grid, G_OH, b_OOH, b_O, k_ads, K_ads, k_OOH, k_O, k_OH, k_free, gamma_mol
    )

    jL = levich_current_density(rpm)
    j_disk = disk_current_density(jk, jL)

    fig, axes = plt.subplots(1, 3, figsize=(3 * l, l))

    draw_free_energy_on_axis(axes[0], G_OH, b_OOH, b_O)

    j_disk_mA = 1000.0 * j_disk
    E_half, j_half = half_wave_potential(U_grid, j_disk_mA)

    axes[1].plot(U_grid, j_disk_mA, color=colors[0])
    axes[1].axhline(1000.0 * jL, color=colors[4], linestyle=":", linewidth=1)

    if E_half is not None:
        axes[1].axhline(j_half, color=colors[3], linestyle="--", linewidth=1)
        axes[1].axvline(E_half, color=colors[3], linestyle="--", linewidth=1)
        axes[1].plot(E_half, j_half, marker="o", color=colors[3])
        axes[1].text(
            E_half,
            j_half,
            f"  E$_{{1/2}}$ = {E_half:.3f} V",
            va="bottom",
            ha="left",
        )

    axes[1].set_xlabel("U / V vs RHE")
    axes[1].set_ylabel(r"$j_{\mathrm{disk}}$ / mA cm$^{-2}$")
    axes[1].set_ylim(-10, 0)
    axes[1].set_title(f"RRDE disk current, {rpm} rpm")

    for i, label in enumerate(coverage_labels):
        axes[2].plot(U_grid, theta[:, i], label=label)

    axes[2].set_xlabel("U / V vs RHE")
    axes[2].set_ylabel("Coverage")
    axes[2].set_ylim(0, 1)
    axes[2].legend()
    axes[2].set_title("Surface coverages")

    plt.tight_layout()
    plt.show()


interact(
    plot_summary_dashboard,
    G_OH=FloatSlider(
        value=default_params["G_OH"],
        min=0.50,
        max=1.10,
        step=0.01,
        description="G_OH",
        continuous_update=False,
    ),
    b_OOH=FloatSlider(
        value=default_params["b_OOH"],
        min=2.50,
        max=3.50,
        step=0.01,
        description="b_OOH",
        continuous_update=False,
    ),
    b_O=FloatSlider(
        value=default_params["b_O"],
        min=-0.80,
        max=0.20,
        step=0.01,
        description="b_O",
        continuous_update=False,
    ),
    k_ads=FloatLogSlider(
        value=default_params["k_ads"],
        base=10,
        min=1,
        max=5,
        step=0.1,
        description="k_ads",
        continuous_update=False,
    ),
    K_ads=FloatSlider(
        value=default_params["K_ads"],
        min=0.05,
        max=3.0,
        step=0.05,
        description="K_ads",
        continuous_update=False,
    ),
    k_OOH=FloatLogSlider(
        value=default_params["k_OOH"],
        base=10,
        min=-2,
        max=3,
        step=0.1,
        description="k_OOH",
        continuous_update=False,
    ),
    k_O=FloatLogSlider(
        value=default_params["k_O"],
        base=10,
        min=-3,
        max=3,
        step=0.1,
        description="k_O",
        continuous_update=False,
    ),
    k_OH=FloatLogSlider(
        value=default_params["k_OH"],
        base=10,
        min=-3,
        max=3,
        step=0.1,
        description="k_OH",
        continuous_update=False,
    ),
    k_free=FloatLogSlider(
        value=default_params["k_free"],
        base=10,
        min=-2,
        max=4,
        step=0.1,
        description="k_free",
        continuous_update=False,
    ),
    gamma=FloatSlider(
        value=default_params["Gamma"] / 1e-12,
        min=10.0,
        max=2000.0,
        step=10.0,
        description="gamma",
        continuous_update=False,
    ),
    rpm=Dropdown(
        options=[400, 900, 1600, 2500, 3600],
        value=1600,
        description="rpm",
    ),
);

interactive(children=(FloatSlider(value=0.82, continuous_update=False, description='G_OH', max=1.1, min=0.5, s…

#Summary

After completing this notebook, you should be able to:


1. Set up a microkinetic model from thermodynamic parameters.

2. Solve for steady-state coverages. Write the mass balance equations for five surface species (free site, O₂*, OOH*, O*, OH*) as a linear system by setting each coverage time derivative to zero and replacing one equation with the site balance. Solve with scipy.linalg.solve and repeat across a potential sweep to get coverage-potential curves.
3. Calculate the kinetic current from the sum of net rates of the four electron-transfer steps weighted by the Faraday constant and the active-site concentration (mol cm⁻²). Apply the Levich equation to estimate the diffusion-limited current at a given rotation rate, and combine both with the Koutecky-Levich relation to get the measurable disk polarization curve.
4. Extract the half-wave potential. Locate the potential at which the disk current is halfway between its limiting value and zero by linear interpolation, and annotate it on the polarization curve.
5. Use interactive sliders to build physical intuition. Adjust thermodynamic parameters (G_OH, b_OOH, b_O) and observe how the free energy landscape changes and how those changes propagate to the surface coverages and current. Adjust kinetic parameters to identify which step is rate-limiting. Change the rotation rate to observe the transition from kinetic to transport control, and observe how Γ (active-site concentration) scales the current without changing the shape of the polarization curve.